# 🛡️ LLM API Error Handling

**Build resilient production-ready LLM applications**

---

## 📋 Overview

**What you'll learn:**
- Common API errors and fixes
- Retry strategies with exponential backoff
- Rate limit handling
- Timeout management
- Circuit breaker pattern
- Fallback strategies

**Time estimate:** ⏱️ 45 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI, RateLimitError, APITimeoutError, APIError
import anthropic
import time
import os
from typing import Optional
from datetime import datetime, timedelta

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🚨 Common LLM API Errors

### 1. Rate Limit Errors
```
RateLimitError: You exceeded your rate limit
```
**Solution**: Retry with exponential backoff

### 2. Timeout Errors
```
APITimeoutError: Request timed out
```
**Solution**: Increase timeout or retry

### 3. Invalid Request
```
BadRequestError: Invalid parameter
```
**Solution**: Validate inputs before calling

### 4. Authentication Errors
```
AuthenticationError: Invalid API key
```
**Solution**: Check API key configuration

### 5. Server Errors
```
InternalServerError: 500 error
```
**Solution**: Retry with backoff

## 🔄 Retry with Exponential Backoff

In [ ]:
def call_llm_with_retry(
    prompt: str,
    max_retries: int = 3,
    base_delay: float = 1.0,
    model: str = "gpt-3.5-turbo"
) -> Optional[str]:
    """Call LLM with exponential backoff retry."""
    
    for attempt in range(max_retries):
        try:
            print(f"  Attempt {attempt + 1}/{max_retries}...")
            
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                timeout=10  # 10 second timeout
            )
            
            print(f"  ✅ Success on attempt {attempt + 1}")
            return response.choices[0].message.content
        
        except RateLimitError as e:
            print(f"  ⚠️ Rate limit hit: {e}")
            if attempt < max_retries - 1:
                # Exponential backoff: 1s, 2s, 4s, 8s...
                delay = base_delay * (2 ** attempt)
                print(f"  Waiting {delay}s before retry...")
                time.sleep(delay)
            else:
                print(f"  ❌ Max retries reached")
                raise
        
        except APITimeoutError as e:
            print(f"  ⚠️ Timeout: {e}")
            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                print(f"  Waiting {delay}s before retry...")
                time.sleep(delay)
            else:
                print(f"  ❌ Max retries reached")
                raise
        
        except APIError as e:
            print(f"  ⚠️ API Error: {e}")
            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                print(f"  Waiting {delay}s before retry...")
                time.sleep(delay)
            else:
                print(f"  ❌ Max retries reached")
                raise
        
        except Exception as e:
            print(f"  ❌ Unexpected error: {e}")
            raise
    
    return None

# Test it
result = call_llm_with_retry("Say hello!")
print(f"\nFinal result: {result}")

## ⏱️ Timeout Management

In [ ]:
def call_with_timeout(prompt: str, timeout_seconds: int = 30):
    """Call LLM with custom timeout."""
    
    try:
        print(f"Calling API with {timeout_seconds}s timeout...")
        
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=100,
            timeout=timeout_seconds
        )
        
        return response.choices[0].message.content
    
    except APITimeoutError:
        print(f"⚠️ Request timed out after {timeout_seconds}s")
        return None

# Test with different timeouts
result = call_with_timeout("Hello!", timeout_seconds=10)
print(f"Result: {result}")

## 🔄 Circuit Breaker Pattern

**Prevents cascading failures**

States:
- ✅ **Closed**: Normal operation
- ⚠️ **Open**: Too many failures, reject requests
- 🔄 **Half-Open**: Test if service recovered

In [ ]:
from enum import Enum
from collections import deque

class CircuitState(Enum):
    CLOSED = "closed"      # Normal
    OPEN = "open"          # Failing, reject requests
    HALF_OPEN = "half_open"  # Testing recovery

class CircuitBreaker:
    """Circuit breaker for LLM API calls."""
    
    def __init__(
        self,
        failure_threshold: int = 5,
        recovery_timeout: int = 60,
        expected_exception: type = Exception
    ):
        self.failure_threshold = failure_threshold
        self.recovery_timeout = recovery_timeout
        self.expected_exception = expected_exception
        
        self.failure_count = 0
        self.last_failure_time = None
        self.state = CircuitState.CLOSED
    
    def call(self, func, *args, **kwargs):
        """Execute function with circuit breaker."""
        
        # Check if we should transition to HALF_OPEN
        if self.state == CircuitState.OPEN:
            if self._should_attempt_reset():
                self.state = CircuitState.HALF_OPEN
                print("  🔄 Circuit HALF_OPEN: Testing recovery...")
            else:
                raise Exception(f"Circuit breaker OPEN. Try again in {self._time_until_retry():.0f}s")
        
        try:
            # Call the function
            result = func(*args, **kwargs)
            
            # Success! Reset if needed
            if self.state == CircuitState.HALF_OPEN:
                self._reset()
                print("  ✅ Circuit CLOSED: Service recovered")
            
            return result
        
        except self.expected_exception as e:
            self._record_failure()
            raise e
    
    def _record_failure(self):
        """Record a failure."""
        self.failure_count += 1
        self.last_failure_time = datetime.now()
        
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
            print(f"  ⚠️ Circuit OPEN: {self.failure_count} failures")
    
    def _should_attempt_reset(self) -> bool:
        """Check if enough time passed to try again."""
        if self.last_failure_time is None:
            return True
        
        elapsed = (datetime.now() - self.last_failure_time).total_seconds()
        return elapsed >= self.recovery_timeout
    
    def _time_until_retry(self) -> float:
        """Time until we can retry."""
        if self.last_failure_time is None:
            return 0
        
        elapsed = (datetime.now() - self.last_failure_time).total_seconds()
        return max(0, self.recovery_timeout - elapsed)
    
    def _reset(self):
        """Reset circuit breaker."""
        self.failure_count = 0
        self.state = CircuitState.CLOSED

# Example usage
breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=10)

def api_call(prompt: str):
    """Simulated API call."""
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50
    )
    return response.choices[0].message.content

# Test it
try:
    result = breaker.call(api_call, "Hello!")
    print(f"Success: {result}")
except Exception as e:
    print(f"Error: {e}")

## 🔀 Fallback Strategies

In [ ]:
class LLMWithFallback:
    """LLM caller with fallback providers."""
    
    def __init__(self):
        self.openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
    
    def call_with_fallback(self, prompt: str) -> dict:
        """
        Try multiple providers in order:
        1. GPT-3.5 (fast, cheap)
        2. Claude (fallback)
        3. GPT-4 (last resort)
        """
        
        # Try GPT-3.5 first
        try:
            print("  Trying GPT-3.5...")
            response = self.openai_client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100,
                timeout=5
            )
            return {
                'provider': 'OpenAI GPT-3.5',
                'result': response.choices[0].message.content,
                'cost': 'low'
            }
        except Exception as e:
            print(f"  ⚠️ GPT-3.5 failed: {e}")
        
        # Fallback to Claude
        try:
            print("  Trying Claude Haiku...")
            response = self.anthropic_client.messages.create(
                model="claude-3-5-haiku-20241022",
                max_tokens=100,
                messages=[{"role": "user", "content": prompt}]
            )
            return {
                'provider': 'Anthropic Claude Haiku',
                'result': response.content[0].text,
                'cost': 'medium'
            }
        except Exception as e:
            print(f"  ⚠️ Claude failed: {e}")
        
        # Last resort: GPT-4
        try:
            print("  Trying GPT-4 (last resort)...")
            response = self.openai_client.chat.completions.create(
                model="gpt-4",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100
            )
            return {
                'provider': 'OpenAI GPT-4',
                'result': response.choices[0].message.content,
                'cost': 'high'
            }
        except Exception as e:
            print(f"  ❌ All providers failed")
            raise Exception("All LLM providers failed") from e

# Test it
fallback_caller = LLMWithFallback()

result = fallback_caller.call_with_fallback("Translate 'Hello' to Spanish")
print(f"\n✅ Provider: {result['provider']}")
print(f"Result: {result['result']}")
print(f"Cost: {result['cost']}")

## 📊 Production-Ready Error Handler

In [ ]:
import logging
from dataclasses import dataclass
from typing import Callable, Any

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@dataclass
class RetryConfig:
    max_retries: int = 3
    base_delay: float = 1.0
    max_delay: float = 60.0
    timeout: int = 30

class ProductionLLMCaller:
    """Production-ready LLM caller with comprehensive error handling."""
    
    def __init__(self, config: RetryConfig = None):
        self.config = config or RetryConfig()
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.circuit_breaker = CircuitBreaker(
            failure_threshold=5,
            recovery_timeout=60
        )
    
    def call(
        self,
        prompt: str,
        model: str = "gpt-3.5-turbo",
        **kwargs
    ) -> Optional[str]:
        """Production LLM call with full error handling."""
        
        def _make_call():
            for attempt in range(self.config.max_retries):
                try:
                    logger.info(f"Attempt {attempt + 1}/{self.config.max_retries}")
                    
                    response = self.client.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt}],
                        timeout=self.config.timeout,
                        **kwargs
                    )
                    
                    logger.info("✅ Call successful")
                    return response.choices[0].message.content
                
                except RateLimitError as e:
                    logger.warning(f"Rate limit hit: {e}")
                    if attempt < self.config.max_retries - 1:
                        delay = min(
                            self.config.base_delay * (2 ** attempt),
                            self.config.max_delay
                        )
                        logger.info(f"Waiting {delay}s...")
                        time.sleep(delay)
                    else:
                        raise
                
                except APITimeoutError as e:
                    logger.warning(f"Timeout: {e}")
                    if attempt < self.config.max_retries - 1:
                        delay = min(
                            self.config.base_delay * (2 ** attempt),
                            self.config.max_delay
                        )
                        time.sleep(delay)
                    else:
                        raise
                
                except Exception as e:
                    logger.error(f"Unexpected error: {e}")
                    raise
            
            return None
        
        # Use circuit breaker
        try:
            return self.circuit_breaker.call(_make_call)
        except Exception as e:
            logger.error(f"Final error: {e}")
            return None

# Test it
caller = ProductionLLMCaller()
result = caller.call("Say hello!")
print(f"\nResult: {result}")

## ✅ Summary

### Error Handling Strategies:

1. **🔄 Retry Logic**
   - Exponential backoff: 1s, 2s, 4s, 8s
   - Max retries: 3-5
   - Cap max delay: 60s

2. **⏱️ Timeout Management**
   - Set reasonable timeouts (10-30s)
   - Different timeouts for different operations
   - Fail fast on timeout

3. **🔄 Circuit Breaker**
   - Prevent cascading failures
   - States: CLOSED → OPEN → HALF_OPEN
   - Recovery timeout: 60s

4. **🔀 Fallback Strategies**
   - Multiple providers
   - Cached responses
   - Degraded functionality

### Best Practices:

```python
# ✅ Good: Comprehensive error handling
try:
    result = call_with_retry(
        prompt,
        max_retries=3,
        timeout=30
    )
except RateLimitError:
    # Handle rate limit
except TimeoutError:
    # Handle timeout
except Exception as e:
    logger.error(f"Error: {e}")
    # Fallback
```

### Production Checklist:
- ✅ Retry with exponential backoff
- ✅ Set timeouts on all API calls
- ✅ Log all errors with context
- ✅ Implement circuit breaker
- ✅ Have fallback providers
- ✅ Monitor error rates
- ✅ Alert on high failure rates

### Common Mistakes:
- ❌ No retry logic
- ❌ Infinite retries
- ❌ No timeout set
- ❌ Swallowing errors silently
- ❌ Same delay for all retries

### Next: `02_llm_basics/06_comparing_providers.ipynb`